<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/02-cuando-el-arbol-se-aprende-todo/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.2 · Cuando el árbol se aprende todo

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas cinco celdas. Dejan listas `despachos`, `entreno`, `validacion` y `prueba`. Deben quedar `20482`, `10034` y `11321` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_despachos(despachos):
    """Deja los datos que se conocen al preparar un despacho: textos como 0 y 1, mes y día."""
    fecha = pd.to_datetime(despachos["fecha"])
    x = pd.get_dummies(despachos[["ciudad", "tipo", "transportista"]], dtype=int)
    x = x.drop(columns=["ciudad_Lima", "tipo_Bodega", "transportista_Propio"], errors="ignore")
    x["mes"] = fecha.dt.month
    x["dia"] = fecha.dt.dayofweek
    return x.join(despachos[["distancia_km", "lineas", "presentaciones"]])

In [3]:
def parte_encontrada(periodo, real, probabilidad, vigilar=0.10):
    """Parte de los casos reales hallada al vigilar, en cada periodo, a los de mayor riesgo."""
    orden = pd.DataFrame({"periodo": periodo, "real": real, "probabilidad": probabilidad})
    orden["azar"] = np.random.default_rng(0).random(len(orden))
    orden = orden.sort_values(["periodo", "probabilidad", "azar"], ascending=[True, False, True])
    puesto = orden.groupby("periodo").cumcount()
    cupo = (orden.groupby("periodo")["real"].transform("size") * vigilar).round().clip(lower=1)
    return orden.loc[puesto < cupo, "real"].sum() / orden["real"].sum()

In [4]:
def encontrados(modelo, tabla):
    """Parte de los tardíos que se encuentra vigilando el 10 % de los despachos de cada día."""
    prob = modelo.predict_proba(preparar_despachos(tabla))[:, 1]
    return parte_encontrada(tabla["fecha"], tabla["llego_tarde"], prob)

In [5]:
despachos = pd.read_csv(DATOS + "despachos.csv")

entreno = despachos[despachos["fecha"] < "2025-01-01"]
validacion = despachos[despachos["fecha"].between("2025-01-01", "2025-06-30")]
prueba = despachos[despachos["fecha"] >= "2025-07-01"]

print(len(entreno), len(validacion), len(prueba))

20482 10034 11321


### 1. ¿Para qué cada grupo?

Responde en un comentario para qué sirve cada uno: `entreno`, `validacion` y `prueba`. Y una más: ¿por qué no elegir el tamaño del árbol mirando `prueba`?

In [6]:
# entreno: el modelo aprende. validacion: elegimos el tamaño. prueba: medimos una sola vez.
# Si elegimos mirando la prueba, el número ya no representa datos nuevos.

### 2. Un árbol pequeño

Entrena un árbol de profundidad 3 con `entreno`, usando `random_state=0`, y guárdalo en `pequeno`. ¿Qué parte de los tardíos encuentra en `validacion`? Debe dar `0.249`.

In [7]:
pequeno = DecisionTreeClassifier(max_depth=3, random_state=0)
pequeno.fit(preparar_despachos(entreno), entreno["llego_tarde"])

print(round(encontrados(pequeno, validacion), 3))

0.249


### 3. Un árbol sin límite

Entrena un árbol sin `max_depth`, en `sin_limite`. Calcula qué parte encuentra en `entreno` y en `validacion`. ¿Qué pasa?

In [8]:
sin_limite = DecisionTreeClassifier(random_state=0)
sin_limite.fit(preparar_despachos(entreno), entreno["llego_tarde"])

print(round(encontrados(sin_limite, entreno), 3))
print(round(encontrados(sin_limite, validacion), 3))

# En entrenamiento llega al 75 %, lo máximo que permite el cupo. En validación baja a cerca de 22 %:
# se aprendió las respuestas.

0.752
0.223


### 4. Otros tamaños

Calcula la parte encontrada en `validacion` para las profundidades 4, 6 y 12. Guárdalas en un diccionario llamado `por_profundidad`. ¿Cuál queda más alta?

In [9]:
por_profundidad = {}
for profundidad in [4, 6, 12]:
    arbol = DecisionTreeClassifier(max_depth=profundidad, random_state=0)
    arbol.fit(preparar_despachos(entreno), entreno["llego_tarde"])
    por_profundidad[profundidad] = round(float(encontrados(arbol, validacion)), 3)

por_profundidad

{4: 0.27, 6: 0.31, 12: 0.245}

### 5. Otro cupo

Si solo se pudiera vigilar el 5 % de cada día, ¿qué parte encontraría el árbol de profundidad 6? Entrena con `entreno` y mide en `validacion`.

> 💡 `prob = arbol.predict_proba(preparar_despachos(validacion))[:, 1]`, y luego `parte_encontrada(validacion["fecha"], validacion["llego_tarde"], prob, vigilar=0.05)`. Debe dar `0.204`.

In [10]:
arbol = DecisionTreeClassifier(max_depth=6, random_state=0)
arbol.fit(preparar_despachos(entreno), entreno["llego_tarde"])
prob = arbol.predict_proba(preparar_despachos(validacion))[:, 1]

print(
    round(parte_encontrada(validacion["fecha"], validacion["llego_tarde"], prob, vigilar=0.05), 3)
)

0.204


### 6. ¿Por qué es demasiado optimista?

Responde en un comentario. Un compañero prueba con 20 tamaños mirando `prueba` y anuncia el mejor número. ¿Por qué ese número es demasiado optimista?

In [11]:
# De 20 intentos, alguno sale alto solo por azar. Al elegir el mejor, se elige también esa suerte.
# Y ya no quedan datos sin mirar para comprobarlo.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.3: Random Forest: ¿vale la pena?**

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0